In [1]:
import pandas as pd
import numpy as np

import torch
from torch import nn
from torch import optim
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_squared_error, r2_score

In [2]:
FILE_PATH_TRAIN = r"D:\KIEMCOM\HK1-N3\ML\Lab1\INTROML\data\split\train\train.csv"
df_train = pd.read_csv(FILE_PATH_TRAIN)
FILE_PATH_TEST = r"D:\KIEMCOM\HK1-N3\ML\Lab1\INTROML\data\split\test\test.csv"
df_test = pd.read_csv(FILE_PATH_TEST)

In [3]:
region_map = {
    'Khu vực 1': 1,
    'Khu vực 2': 2,
    'Khu vực 3': 3,
    'Khác': 0
}

df_train['region_encoded'] = df_train['region'].map(region_map)
df_test['region_encoded'] = df_test['region'].map(region_map)

In [4]:
# *** BƯỚC 1: LỌC DỮ LIỆU - LOẠI BỎ OUTLIERS ***
# Đây là các ranh giới "hợp lý" (bạn có thể điều chỉnh nếu muốn)
sane_bedrooms = 10
sane_bathrooms = 10
sane_area = 500  # (m2)
sane_price = 100 # (tỷ)

# Lọc tập train
df_train = df_train[
    (df_train['bedrooms'] < sane_bedrooms) &
    (df_train['bathrooms'] < sane_bathrooms) &
    (df_train['area'] < sane_area) &
    (df_train['price'] < sane_price) &
    (df_train['price'] > 0.1) # Loại bỏ giá quá thấp
]

# Lọc tập test
df_test = df_test[
    (df_test['bedrooms'] < sane_bedrooms) &
    (df_test['bathrooms'] < sane_bathrooms) &
    (df_test['area'] < sane_area) &
    (df_test['price'] < sane_price) &
    (df_test['price'] > 0.1) # Loại bỏ giá quá thấp
]

# *** BƯỚC 2: CHẠY LẠI Y HỆT CODE CŨ CỦA BẠN ***
# (Toàn bộ code từ bước tách X_train_orig, y_train_orig,
# log-transform X và Y, Scaler, Training, và Đánh giá)

# 1. Tách X, y gốc (từ df_train, df_test đã lọc)
X_train_orig = df_train[['area', 'bedrooms', 'bathrooms', 'region_encoded']]
y_train_orig = df_train['price']
X_test_orig = df_test[['area', 'bedrooms', 'bathrooms', 'region_encoded']]
y_test_orig = df_test['price']

In [5]:
# 2. *** LOG-TRANSFORM CẢ X (các cột bị lệch) VÀ Y ***

# Các cột bị lệch (skewed)
skewed_features = ['area', 'bedrooms', 'bathrooms']
# Cột danh mục (giữ nguyên)
categorical_feature = ['region_encoded']

# Áp dụng log1p cho các cột X bị lệch
X_train_log_skewed = np.log1p(X_train_orig[skewed_features])
X_test_log_skewed = np.log1p(X_test_orig[skewed_features])

# Reset index để concat không bị lỗi (quan trọng)
X_train_log_skewed.reset_index(drop=True, inplace=True)
X_test_log_skewed.reset_index(drop=True, inplace=True)
X_train_orig.reset_index(drop=True, inplace=True)
X_test_orig.reset_index(drop=True, inplace=True)

# Ghép lại X_log_skewed với cột region_encoded (không bị log)
X_train = pd.concat([X_train_log_skewed, X_train_orig[categorical_feature]], axis=1)
X_test = pd.concat([X_test_log_skewed, X_test_orig[categorical_feature]], axis=1)

# Áp dụng log1p cho Y
y_train = np.log1p(y_train_orig)
y_test = np.log1p(y_test_orig)

In [6]:
# 3. *** SCALER (trên dữ liệu đã log-transform) ***
# Bây giờ StandardScaler sẽ hoạt động tốt vì outliers đã bị "nén" lại

scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train)
X_test_s = scaler_X.transform(X_test)

y_train_reshaped = y_train.values.reshape(-1, 1)
y_test_reshaped = y_test.values.reshape(-1, 1)

scaler_y = StandardScaler()
y_train_s = scaler_y.fit_transform(y_train_reshaped)
y_test_s = scaler_y.transform(y_test_reshaped)

# 4. *** PYTORCH & TRAINING (Giữ nguyên) ***
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32) 
X_test_t = torch.tensor(X_test_s, dtype=torch.float32)
y_test_t = torch.tensor(y_test_s, dtype=torch.float32)

# ... (Phần code X_train_t, y_train_t của bạn giữ nguyên) ...

model = nn.Linear(X_train_t.shape[1], 1)
criterion = nn.MSELoss()

# *** THAY ĐỔI Ở ĐÂY ***
optimizer = optim.Adam(model.parameters(), lr=0.001) # Giảm lr xuống 0.001
epochs = 10000 # Tăng epochs lên 10,000 (hoặc 20,000)
# **********************

train_losses = [] # Lưu lại loss để xem

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    y_pred_train = model(X_train_t)
    loss = criterion(y_pred_train, y_train_t)
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item()) # Lưu loss
    
    # Bật lại print(loss)
    if (epoch+1) % 1000 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {loss.item():.6f}")

# ... (Phần code đánh giá giữ nguyên) ...

# ... (Sau khi vòng lặp training kết thúc) ...

model.eval()

# *** 1. KIỂM TRA TRÊN TẬP TRAIN ***
with torch.no_grad():
    # Dùng X_train_t
    y_pred_train_t = model(X_train_t) 
    
    # (Logic inverse_transform y hệt như test)
    y_pred_scaled_log = y_pred_train_t.cpu().numpy()
    y_pred_log = scaler_y.inverse_transform(y_pred_scaled_log)
    y_pred_train_orig = np.expm1(y_pred_log)
    
    # Dùng y_train_orig (dataframe gốc trước khi log)
    r2_train = r2_score(y_train_orig, y_pred_train_orig)
    print(f"R2 TRÊN TẬP TRAIN: {r2_train:.4f}")

# *** 2. KIỂM TRA TRÊN TẬP TEST (Code cũ của bạn) ***
with torch.no_grad():
    y_pred_t = model(X_test_t)
    y_pred_scaled_log = y_pred_t.cpu().numpy()
    y_pred_log = scaler_y.inverse_transform(y_pred_scaled_log)
    y_pred_orig = np.expm1(y_pred_log)

# So sánh GỐC vs GỐC
r2 = r2_score(y_test_orig, y_pred_orig)
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
print(f"R2 TRÊN TẬP TEST: {r2:.4f}, RMSE: {rmse:.4f}")
    
# 5. *** ĐÁNH GIÁ (Giữ nguyên logic inverse_transform) ***
model.eval()
with torch.no_grad():
    y_pred_t = model(X_test_t)
    y_pred_scaled_log = y_pred_t.cpu().numpy()
    
    # Inverse scale (ra giá trị log)
    y_pred_log = scaler_y.inverse_transform(y_pred_scaled_log)
    
    # Inverse log (ra giá trị gốc)
    y_pred_orig = np.expm1(y_pred_log)

# So sánh GỐC vs GỐC
r2 = r2_score(y_test_orig, y_pred_orig)
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
print(f"R2 (đã sửa log-transform X và Y): {r2:.4f}, RMSE (đã sửa log-transform X và Y): {rmse:.4f}")
    

Epoch [1000/10000] - Loss: 0.668328
Epoch [2000/10000] - Loss: 0.665574
Epoch [3000/10000] - Loss: 0.665574
Epoch [4000/10000] - Loss: 0.665574
Epoch [5000/10000] - Loss: 0.665574
Epoch [6000/10000] - Loss: 0.665574
Epoch [7000/10000] - Loss: 0.665574
Epoch [8000/10000] - Loss: 0.665574
Epoch [9000/10000] - Loss: 0.665574
Epoch [10000/10000] - Loss: 0.665574
R2 TRÊN TẬP TRAIN: 0.2517
R2 TRÊN TẬP TEST: 0.3574, RMSE: 7.8729
R2 (đã sửa log-transform X và Y): 0.3574, RMSE (đã sửa log-transform X và Y): 7.8729
